In [1]:
import pandas as pd
import numpy as np
import os

RAW = '../data/raw/'
PROCESSED = '../data/processed/'
FINAL = '../data/final/'

In [2]:
#Cargamos archivo de valor catastral
df_cat_raw = pd.read_excel(RAW + 'valor_catastral.xlsx', header=None)

print(f'Dimensiones: {df_cat_raw.shape}')
print(df_cat_raw.iloc[:8, :5].to_string())

C:\Users\raned\anaconda3\envs\environment_uoc20232pec3-actual\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Dimensiones: (773, 5)
                                                                       0           1                 2                             3                                 4
0                                              E. Edificación y Vivienda         NaN               NaN                           NaN                               NaN
1                                  Estadística Registral Catastral (IBI)         NaN               NaN                           NaN                               NaN
2  5.1.C. Valor catastral medio de Uso Residencial por Distrito y Barrio         NaN               NaN                           NaN                               NaN
3                                                                    NaN         NaN               NaN  Valor catastral por inmueble  Valor catastral medio por metro2
4                                                                   2020      Madrid  Ciudad de Madrid                    119.045,78           

In [3]:
#Función para limpiar valores con puntos de miles y comas decimales
def limpiar_catastral(valor):
    if pd.isna(valor):
        return np.nan
    if str(valor).strip() in ['-', '']:
        return np.nan
    try:
        # Formato español: 119.045,78 → 119045.78
        return float(str(valor).replace('.', '').replace(',', '.').strip())
    except:
        return np.nan

#Ver años disponibles
años = sorted(set(
    str(row) for row in df_cat_raw.iloc[4:, 0]
    if str(row).strip().isdigit()))
print(f'Años disponibles: {años}')

#Procesamos solo 2023, barrios reales
registros = []
for idx, row in df_cat_raw.iterrows():
    año = str(row[0]).strip()
    if año != '2023':
        continue
    
    barrio = str(row[2]).strip()
    codigo = barrio.split('.')[0].strip()
    if not (len(codigo) == 3 and codigo.isdigit()):
        continue
    
    registros.append({
        'cod_barrio': codigo,
        'barrio': barrio,
        'catastral_inmueble': limpiar_catastral(row[3]),
        'catastral_m2': limpiar_catastral(row[4])
    })

df_catastral = pd.DataFrame(registros)

print(f'\nDimensiones: {df_catastral.shape}')
print(f'Valores nulos:\n{df_catastral.isnull().sum()}')
print(df_catastral.head(10).to_string())

Años disponibles: ['2020', '2021', '2022', '2023', '2024']

Dimensiones: (131, 4)
Valores nulos:
cod_barrio            0
barrio                0
catastral_inmueble    0
catastral_m2          0
dtype: int64
  cod_barrio            barrio  catastral_inmueble  catastral_m2
0        011      011. Palacio           138151.08       1377.18
1        012  012. Embajadores            97607.50       1260.41
2        013       013. Cortes           174473.00       1546.72
3        014     014. Justicia           202105.64       1642.98
4        015  015. Universidad           123074.89       1389.58
5        016          016. Sol           192386.03       1453.18
6        021     021. Imperial           124427.33       1291.00
7        022      022. Acacias           118759.78       1257.57
8        023      023. Chopera            93634.22       1173.47
9        024      024. Legazpi           150226.83       1403.66


In [4]:
#Calculamos variación catastral 2020-2023
registros_2020 = []
for idx, row in df_cat_raw.iterrows():
    año = str(row[0]).strip()
    if año != '2020':
        continue
    
    barrio = str(row[2]).strip()
    codigo = barrio.split('.')[0].strip()
    if not (len(codigo) == 3 and codigo.isdigit()):
        continue
    
    registros_2020.append({
        'cod_barrio': codigo,
        'catastral_m2_2020': limpiar_catastral(row[4])})

df_cat_2020 = pd.DataFrame(registros_2020)

#unimow y calculamos variación
df_catastral = df_catastral.merge(df_cat_2020, on='cod_barrio')

df_catastral['var_catastral_m2_2023'] = (
    (df_catastral['catastral_m2'] - df_catastral['catastral_m2_2020'])
    / df_catastral['catastral_m2_2020'] * 100).round(2)

#selecció de columnas finales
df_catastral_final = df_catastral[[
    'cod_barrio',
    'barrio',
    'catastral_m2',
    'var_catastral_m2_2023']].copy()


print(f'Dimensiones: {df_catastral_final.shape}')
print(f'Valores nulos:\n{df_catastral_final.isnull().sum()}')
print(f'\nTop 10 barrios mayor valor catastral:')
print(df_catastral_final[['barrio', 'catastral_m2']]
      .sort_values('catastral_m2', ascending=False)
      .head(10).to_string())

print(f'\nTop 10 mayor variación catastral 2020-2023:')
print(df_catastral_final[['barrio', 'var_catastral_m2_2023']]
      .sort_values('var_catastral_m2_2023', ascending=False)
      .head(10).to_string())

#Guardamos en carpeta de processed
df_catastral_final.to_csv(PROCESSED + 'df_catastral.csv', index=False)

Dimensiones: (131, 4)
Valores nulos:
cod_barrio               0
barrio                   0
catastral_m2             0
var_catastral_m2_2023    0
dtype: int64

Top 10 barrios mayor valor catastral:
                barrio  catastral_m2
17  035. Los Jerónimos       2185.70
19      041. Recoletos       2056.80
25        051. El Viso       2045.95
24     046. Castellana       2012.75
52      092. Argüelles       1882.39
18     036. Niño Jesús       1741.10
29   055. Nueva España       1703.74
16          034. Ibiza       1683.72
20           042. Goya       1673.54
42   076. Vallehermoso       1670.09

Top 10 mayor variación catastral 2020-2023:
                                barrio  var_catastral_m2_2023
127                    212. Aeropuerto                  16.66
117                  194. El Cañaveral                   2.71
35                  065. Valdeacederas                   2.09
41                     075. Ríos Rosas                   0.84
110                       175. Ángeles   